# Demonstration of rotationally symmetric SVD decomposition in Panther-EM.

1. Download and simulate cryo-EM projections of a structure
2. Perform optimized SVD decomposition by exploiting in-plane rotational symmetry (in polar coordinates)
3. Plot singular values of the decomposition
4. Plot polar/cartesian features of decomposition

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from torch_so3 import get_uniform_euler_angles
from ttsim3d.models import Simulator, SimulatorConfig

from panther_em.decomposition import PolarProjectionDecomposer
from panther_em.inference.projection_reconstruction import ProjectionReconstructor

## Load in volume to take projections from

In [ ]:
!wget -nc https://files.rcsb.org/download/pdb_00007myi.cif

In [ ]:
pdb_path = "pdb_00007myi.cif"

In [ ]:
# Instantiate the configuration object
sim_conf = SimulatorConfig(
    voltage=300.0,  # in keV
    apply_dose_weighting=True,
    dose_start=0.0,  # in e-/A^2
    dose_end=15.0,  # in e-/A^2
    upsampling=-1,  # auto
)

# Instantiate the simulator
sim = Simulator(
    pdb_filepath=pdb_path,
    pixel_spacing=0.4,  # Angstroms
    volume_shape=(512, 512, 512),
    b_factor_scaling=1.0,
    additional_b_factor=30.0,
    simulator_config=sim_conf,
)

In [ ]:
# Run the simulation
volume = sim.run()
print(type(volume))
print(volume.shape)

In [ ]:
# import mrcfile

# # NOTE: Or load in pre-computed volume from elsewhere
# volume_path = "/data/mgiammar/MOSAICS/data/volumes/parsed_6Q8Y_whole_LSU_match3.mrc"
# volume = mrcfile.read(volume_path).data.copy()
# volume = torch.from_numpy(volume).float()
# print(type(volume))
# print(volume.shape)

In [ ]:
volume = volume.numpy()

## Define the out-of-plane orientations

Using `torch-so3` to approximate points over SO(3) space, except all in-plane angles are set to zero.

In [ ]:
angles = get_uniform_euler_angles(psi_step=360.0, theta_step=6.0)
print(angles.shape)

phi_angles = angles[:, 0].numpy()
theta_angles = angles[:, 1].numpy()

## Run the decomposition

In [ ]:
radial_components = 256
angular_components = 360

decomposer = PolarProjectionDecomposer(
    volume=volume,
    phi_values=phi_angles,
    theta_values=theta_angles,
    device="cuda",
    num_radius=radial_components,
    num_angle=angular_components,
)

result = decomposer.do_decomposition()

In [ ]:
result

In [ ]:
fix, ax = result.scree_plot()
plt.yscale("log")

In [ ]:
sing_vals_flat = result.singular_values.flatten()
sing_vals_flat.sort()
sing_vals_flat = sing_vals_flat[::-1]

plt.plot(sing_vals_flat)
plt.yscale("log")
plt.xscale("log")
plt.ylim(1e-1, 1e4)
plt.grid()
plt.show()

In [ ]:
running_sum = (sing_vals_flat**2).cumsum() / (sing_vals_flat**2).sum()

x = np.arange(len(sing_vals_flat))
n = 512 * 512 * 6602
x = x / n

plt.plot(x, 1 - running_sum)
plt.xscale("log")
plt.yscale("log")
plt.grid()
plt.show()

In [ ]:
import matplotlib.figure
import matplotlib.pyplot as plt
import numpy as np

from panther_em.decomposition.result import DecompositionResult


def plot_singular_values(
    result: DecompositionResult,
    k_indices: list[int] | None = None,
    num_eigenvalues: int | None = None,
    log_log: bool = True,
    ax: plt.Axes | None = None,
    cmap: str = "viridis",
) -> matplotlib.figure.Figure:
    """Plot singular values from a decomposition result, optionally on a log-log scale.

    Parameters
    ----------
    result : DecompositionResult
        The decomposition result containing singular values.
    k_indices : list[int] | None
        Which angular frequency indices to plot. If None, plots all k_max.
    num_eigenvalues : int | None
        Number of leading eigenvalues to show per frequency. If None, shows all.
    log_log : bool
        Whether to use log-log axes. Default is True.
    ax : plt.Axes | None
        Existing axes to plot on. If None, creates a new figure.
    cmap : str
        Colormap name for distinguishing frequency indices. Default is "viridis".

    Returns
    -------
    matplotlib.figure.Figure
        The figure containing the plot.
    """
    if k_indices is None:
        k_indices = list(range(result.k_max))
    if num_eigenvalues is None:
        num_eigenvalues = result.num_radial_components

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    else:
        fig = ax.get_figure()

    colormap = plt.get_cmap(cmap)
    colors = [colormap(i / max(len(k_indices) - 1, 1)) for i in range(len(k_indices))]

    eig_indices = np.arange(1, num_eigenvalues + 1)  # 1-indexed for log scale

    for color, k in zip(colors, k_indices, strict=False):
        sv = np.abs(result.singular_values[k, :num_eigenvalues])
        ax.plot(
            eig_indices,
            sv,
            marker=".",
            markersize=3,
            linewidth=0.8,
            color=color,
            label=f"k={k}",
        )

    if log_log:
        ax.set_xscale("log")
        ax.set_yscale("log")

    ax.set_xlabel("Eigenvalue index")
    ax.set_ylabel("Singular value")
    ax.set_title(
        "Singular value spectrum (log-log)" if log_log else "Singular value spectrum"
    )

    # Only show legend if a manageable number of lines
    if len(k_indices) <= 20:
        ax.legend(fontsize="small", ncol=2)
    else:
        sm = plt.cm.ScalarMappable(
            cmap=colormap,
            norm=plt.Normalize(vmin=min(k_indices), vmax=max(k_indices)),
        )
        sm.set_array([])
        fig.colorbar(sm, ax=ax, label="Angular frequency index k")

    fig.tight_layout()

    return fig

In [ ]:
def plot_cumulative_singular_values(
    result: DecompositionResult,
    k_indices: list[int] | None = None,
    num_eigenvalues: int | None = None,
    ax: plt.Axes | None = None,
    cmap: str = "viridis",
) -> matplotlib.figure.Figure:
    """Plot cumulative fractional energy captured by singular values.

    For each angular frequency k, plots the cumulative sum of squared singular
    values normalized by the total, showing what fraction of variance is captured
    by the first N components.

    Parameters
    ----------
    result : DecompositionResult
        The decomposition result containing singular values.
    k_indices : list[int] | None
        Which angular frequency indices to plot. If None, plots all k_max.
    num_eigenvalues : int | None
        Number of leading eigenvalues to show. If None, shows all.
    ax : plt.Axes | None
        Existing axes to plot on. If None, creates a new figure.
    cmap : str
        Colormap name for distinguishing frequency indices. Default is "viridis".

    Returns
    -------
    matplotlib.figure.Figure
        The figure containing the plot.
    """
    if k_indices is None:
        k_indices = list(range(result.k_max))
    if num_eigenvalues is None:
        num_eigenvalues = result.num_radial_components

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    else:
        fig = ax.get_figure()

    colormap = plt.get_cmap(cmap)
    colors = [colormap(i / max(len(k_indices) - 1, 1)) for i in range(len(k_indices))]

    eig_indices = np.arange(1, num_eigenvalues + 1)

    for color, k in zip(colors, k_indices, strict=False):
        sv_sq = np.abs(result.singular_values[k]) ** 2
        total_energy = sv_sq.sum()
        if total_energy == 0:
            continue
        cumulative = np.cumsum(sv_sq[:num_eigenvalues]) / total_energy

        ax.plot(
            eig_indices,
            cumulative,
            marker=".",
            markersize=3,
            linewidth=0.8,
            color=color,
            label=f"k={k}",
        )

    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Number of components")
    ax.set_ylabel("Cumulative fractional energy")
    ax.set_title("Cumulative singular value energy")
    ax.axhline(y=0.95, color="gray", linestyle="--", linewidth=0.8, label="95%")
    ax.axhline(y=0.99, color="gray", linestyle=":", linewidth=0.8, label="99%")

    # Only show legend if a manageable number of lines
    if len(k_indices) <= 20:
        ax.legend(fontsize="small", ncol=2)
    else:
        sm = plt.cm.ScalarMappable(
            cmap=colormap,
            norm=plt.Normalize(vmin=min(k_indices), vmax=max(k_indices)),
        )
        sm.set_array([])
        fig.colorbar(sm, ax=ax, label="Angular frequency index k")

    fig.tight_layout()
    return fig

In [ ]:
# # All frequencies
# fig = plot_singular_values(result, cmap="turbo")
# plt.show()

# logarithmic set of k indices
k_indices = np.logspace(0, np.log10(result.k_max - 1), num=64, dtype=int)
k_indices = np.unique(k_indices)
print(k_indices)

# Specific frequencies
fig = plot_singular_values(result, k_indices=k_indices, cmap="rainbow")
# plt.show()

# # On existing subplots
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# plot_singular_values(result, k_indices=list(range(10)), ax=axes[0])
# plot_singular_values(result, k_indices=list(range(10, 50)), ax=axes[1])
# plt.show()

In [ ]:
# All frequencies
fig = plot_cumulative_singular_values(result)
plt.show()

# Specific frequencies, first 100 components
fig = plot_cumulative_singular_values(
    result, k_indices=k_indices, num_eigenvalues=100, cmap="rainbow"
)
plt.show()

# Side-by-side with the log-log plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_singular_values(result, k_indices=list(range(0, 50, 5)), ax=axes[0])
plot_cumulative_singular_values(result, k_indices=list(range(0, 50, 5)), ax=axes[1])
plt.show()

## Visualize results

In [ ]:
k_idx = 0
eig_idx = 0

polar_feature = reconstructor.construct_polar_feature(k_idx=k_idx, eig_idx=eig_idx)
cartesian_feature = reconstructor.construct_cartesian_feature(k_idx=k_idx, eig_idx=eig_idx)

l2_norm_polar = np.linalg.norm(polar_feature)
l2_norm_cartesian = np.linalg.norm(cartesian_feature)

print(f"L2 norm of polar feature: {l2_norm_polar:.4f}")
print(f"L2 norm of cartesian feature: {l2_norm_cartesian:.4f}")

In [ ]:
# Table of norms across k_idx and eig_idx
norms_table = []
for k_idx in range(min(50, result.k_max)):
    for eig_idx in range(min(50, result.num_radial_components)):
        polar_feature = decomposer.construct_polar_feature(k_idx=k_idx, eig_idx=eig_idx)
        cartesian_feature = decomposer.construct_cartesian_feature(k_idx=k_idx, eig_idx=eig_idx)

        l2_norm_polar = np.linalg.norm(polar_feature)
        l2_norm_cartesian = np.linalg.norm(cartesian_feature)

        norms_table.append((k_idx, eig_idx, l2_norm_polar, l2_norm_cartesian))

# Print out rows nicely formatted with spacing to be consistent and easy to read
for row in norms_table[:10]:
    print(f"k_idx={row[0]:3d}, eig_idx={row[1]:3d}, L2 Polar={row[2]:.4f}, L2 Cartesian={row[3]:.4f}")

In [ ]:
norms_table

In [ ]:
# histogram of polar L2 norm, cartesian L2 norm, and their ratio
polar_norms = [row[2] for row in norms_table]
cartesian_norms = [row[3] for row in norms_table]
ratios = [c / p if p > 0 else 0 for p, c in zip(polar_norms, cartesian_norms)]

plt.figure(figsize=(12, 4))
plt.hist(polar_norms, bins=3, alpha=0.5, label="Polar L2 Norm", density=True)
plt.hist(cartesian_norms, bins=30, alpha=0.5, label="Cartesian L2 Norm", density=True)
# plt.hist(ratios, bins=30, alpha=0.5, label="Cartesian/Polar L2 Norm Ratio")
plt.legend()
plt.title("Distribution of L2 Norms and their Ratio")
plt.xlabel("L2 Norm Value")
plt.ylabel("Frequency")
plt.yscale("log")
plt.show()

In [ ]:
for row in norms_table:
    print(f"k_idx={row[0]:3d}, eig_idx={row[1]:3d}, L2 Polar={row[2]:.4f}, L2 Cartesian={row[3]:.4f}")

In [ ]:
k_idx = 12
eig_idx = 1

polar_feature = decomposer.construct_polar_feature(k_idx=k_idx, eig_idx=eig_idx)
cartesian_feature = decomposer.construct_cartesian_feature(k_idx=k_idx, eig_idx=eig_idx)


# Plot the real and imaginary parts of the polar on a polar mpl plot
fig, ax = plt.subplots(1, 2, figsize=(8, 6))

im0 = ax[0].imshow(np.real(polar_feature), cmap="coolwarm")
ax[0].set_title("Real Part")
plt.colorbar(im0, ax=ax[0])

im1 = ax[1].imshow(np.imag(polar_feature), cmap="coolwarm")
ax[1].set_title("Imaginary Part")
plt.colorbar(im1, ax=ax[1])

plt.tight_layout()
plt.show()

# Plot the reconstructed cartesian feature
fig, ax = plt.subplots(1, 2, figsize=(8, 5))

im0 = ax[0].imshow(np.real(cartesian_feature), cmap="coolwarm")
ax[0].set_title("Real Part")
plt.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)

im1 = ax[1].imshow(np.imag(cartesian_feature), cmap="coolwarm")
ax[1].set_title("Imaginary Part")
plt.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
num_k = 10
num_eig = 8

# Set squeeze=False so axes are always a 2D array even if num_k or num_eig is 1
fig_real, axes_real = plt.subplots(
    num_k, num_eig, figsize=(num_eig * 2.5, num_k * 2.5), squeeze=False,
)
fig_real.suptitle("Cartesian Features - Real Part", fontsize=16)

fig_imag, axes_imag = plt.subplots(
    num_k, num_eig, figsize=(num_eig * 2.5, num_k * 2.5), squeeze=False
)
fig_imag.suptitle("Cartesian Features - Imaginary Part", fontsize=16)

for k in range(num_k):
    for eig in range(num_eig):
        # Retrieve the reconstructed Cartesian feature
        cartesian_feature = decomposer.construct_cartesian_feature(
            k_idx=k, eig_idx=eig
        )

        # Plot Real Part
        ax_r = axes_real[k, eig]
        ax_r.imshow(np.real(cartesian_feature), cmap="coolwarm")
        ax_r.set_xticks([])
        ax_r.set_yticks([])
        if k == 0:
            ax_r.set_title(f"eig_idx={eig}")
        if eig == 0:
            ax_r.set_ylabel(f"k_idx={k}", size="large")

        # Plot Imaginary Part
        ax_i = axes_imag[k, eig]
        ax_i.imshow(np.imag(cartesian_feature), cmap="coolwarm")
        ax_i.set_xticks([])
        ax_i.set_yticks([])
        if k == 0:
            ax_i.set_title(f"eig_idx={eig}")
        if eig == 0:
            ax_i.set_ylabel(f"k_idx={k}", size="large")

fig_real.tight_layout()
fig_imag.tight_layout()

# Save the figures as pdf
plt.figure(fig_real.number)
plt.savefig("cartesian_features_real.pdf", format="pdf", bbox_inches="tight")
plt.figure(fig_imag.number)
plt.savefig("cartesian_features_imag.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# result.left_singular_vectors.shape

In [ ]:
# proj = decomposer.reconstruct_projection(orientation_idx=0)
# proj.shape

In [ ]:
# plt.imshow(proj.real, cmap="gray")
# plt.colorbar()
# plt.title("Reconstructed Projection (Real Part)")
# plt.show()

# plt.imshow(proj.imag, cmap="gray")
# plt.colorbar()
# plt.title("Reconstructed Projection (Imaginary Part)")
# plt.show()

In [ ]:
# # Reconstruct and plot the first N projections
# N = 256

# projections = []
# for i in tqdm.tqdm(range(N)):
#     proj = decomposer.reconstruct_projection(orientation_idx=i)
#     projections.append(proj)

In [ ]:
# # starting index
# start_idx = 32

# fig, ax = plt.subplots(4, 4, figsize=(12, 12))
# for i in range(ax.shape[0]):
#     for j in range(ax.shape[1]):
#         idx = start_idx + i * ax.shape[1] + j
#         ax[i, j].imshow(projections[idx].real, cmap="magma")
#         ax[i, j].set_title(f"Reconstruction {idx}")

# plt.tight_layout()
# plt.show()

In [ ]:
# # Display real and imaginary parts of the projection (with colorbars)
# fig, ax = plt.subplots(1, 2, figsize=(8, 5))

# im0 = ax[0].imshow(np.real(proj))
# ax[0].set_title("Real Part")
# plt.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)

# im1 = ax[1].imshow(np.imag(proj))
# ax[1].set_title("Imaginary Part")
# plt.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04)

# plt.tight_layout()
# plt.show()